In [ ]:
import os

import matplotlib.pyplot as plt
import scanpy as sc
import lamindb as ln

ln.track("bOC16St25Xx0")

In [ ]:
FIGURES_DIR = '/home/workspace/PCD_data/xenium/commercial/figures/processing'

os.makedirs(FIGURES_DIR, exist_ok=True)

In [ ]:
def save_fig(fig, stem, description, ulabels=(), features=None):
    fig.tight_layout()
    for ax in fig.axes:
        for coll in ax.collections:
            coll.set_rasterized(True)
    for ext in ['png', 'pdf']:
        path = os.path.join(FIGURES_DIR, f'{stem}.{ext}')
        fig.savefig(path, dpi=150, bbox_inches='tight')
        artifact = ln.Artifact(path, key=f'test_{stem}.{ext}', description=description).save()
        if ulabels:
            artifact.ulabels.add(*ulabels)
        if features:
            artifact.features.add_values(features)
    plt.close(fig)

In [ ]:
processing_label = ln.ULabel.get(name='processing')
umap_label       = ln.ULabel.get(name='umap')

ln.Param(name='harmony',    dtype=bool).save()
ln.Param(name='n_neighbors', dtype=int).save()
ln.Param(name='n_pcs',      dtype=int).save()
ln.Param(name='spread',     dtype=float).save()

In [ ]:
ln.Artifact.filter(suffix='.h5ad').order_by('-created_at').df()

In [ ]:
combined = ln.Artifact.get('Po61viBGaIukVXxG0000').load()

In [ ]:
SPREADS = list(range(1, 11))  # 1 through 10

fig, axes = plt.subplots(2, 5, figsize=(5 * 5, 5 * 2), squeeze=False)

for ax, spread in zip(axes.ravel(), SPREADS):
    sc.tl.umap(combined, random_state=0, spread=spread)  # overwrites obsm['X_umap']
    sc.pl.umap(combined, color='sample', legend_loc=None,
               ax=ax, show=False, size=4, frameon=False, title=f'spread={spread}')

# save_fig(fig, 'umap_spread_sweep',
#          'UMAP spread sweep (1–10), colored by sample',
#          ulabels=(processing_label, umap_label),
#          features={'harmony': True, 'n_neighbors': 15, 'n_pcs': 30})

# ln.finish()

In [ ]:
SPREADS = [1, 2, 4, 6, 8, 10]

fig, axes = plt.subplots(2, 3, figsize=(5 * 3, 5 * 2), squeeze=False)

for ax, spread in zip(axes.ravel(), SPREADS):
    sc.tl.umap(combined, random_state=0, spread=spread, maxiter=200)  # overwrites obsm['X_umap']
    sc.pl.umap(combined, color='sample', legend_loc=None,
               ax=ax, show=False, size=4, frameon=False, title=f'spread={spread}')

# save_fig(fig, 'umap_spread_sweep',
#          'UMAP spread sweep (1, 2, 4, 6, 8, 10), colored by sample',
#          ulabels=(processing_label, umap_label),
#          features={'harmony': True, 'n_neighbors': 15, 'n_pcs': 30})

# ln.finish()